<a href="https://colab.research.google.com/github/Sukrit-Prakash/Assignment/blob/main/FakeNewsDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# !pip -q install transformers accelerate evaluate scikit-learn
!pip install -U "transformers>=4.42" "accelerate>=0.33" "datasets>=2.20" "evaluate>=0.4"



In [6]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, EarlyStoppingCallback, set_seed)

In [7]:
TRAIN_TSV = "multimodal_train.tsv"  # <-- your path
SEP = "\t"                         # TSV uses tabs
MODEL_NAME = "roberta-base"        # Good speed/quality tradeoff; switch to "microsoft/deberta-v3-base" if you prefer
TEXT_COL_CANDIDATES = ["all_text", "clean_text", "text", "clean_title", "title"]  # we will concat existing
LABEL_PRIORITIES = ["6_way_label", "3_way_label", "2_way_label"]                   # prefer most granular
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
WARMUP_RATIO = 0.08
VAL_SIZE = 0.1
SEED = 42
USE_FP16 = True                    # mixed precision for speed on GPU
PATIENCE = 2                       # early stopping on macro-F1
OUTPUT_DIR = "./fakeddit_text_model"

In [8]:
def set_all_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    set_seed(seed)

set_all_seeds(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [9]:
# ---------------------------
# Load data
# ---------------------------
df = pd.read_csv(TRAIN_TSV, sep=SEP)
print("Columns:", list(df.columns))

# Pick label column by priority (Why: we want the richest supervision available)
label_col = None
for c in LABEL_PRIORITIES:
    if c in df.columns:
        label_col = c
        break
if label_col is None:
    raise ValueError(f"No label column found. Looked for {LABEL_PRIORITIES}")
print("Using label column:", label_col)

Columns: ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label']
Using label column: 6_way_label


In [10]:
# Build a single text column by concatenating existing text columns
usable_text_cols = [c for c in TEXT_COL_CANDIDATES if c in df.columns]
if not usable_text_cols:
    raise ValueError(f"None of the expected text columns found. Tried {TEXT_COL_CANDIDATES}")


In [11]:
text = df[usable_text_cols[0]].astype(str)
for c in usable_text_cols[1:]:
    # Why [SEP]? Gives the model a soft hint that segments are separate
    text = text.str.cat(df[c].astype(str), sep=" [SEP] ")
df["__text__"] = text

In [12]:
# Basic hygiene: drop rows with missing/empty inputs or labels (Why: avoid noisy/invalid samples)
before = len(df)
df = df.dropna(subset=["__text__", label_col])
df = df[df["__text__"].str.strip() != ""]
after = len(df)
print(f"Dropped {before - after} empty rows. Remaining: {after}")

Dropped 0 empty rows. Remaining: 564000


In [27]:
# Build classes as strings
classes = sorted(pd.Series(df[label_col].astype(str)).unique().tolist())

# Clean maps: id2label {int->str}, label2id {str->int}
label2id = {label: int(i) for i, label in enumerate(classes)}
id2label = {int(i): label for label, i in label2id.items()}
num_labels = len(classes)

# IMPORTANT: cast to str before map so keys match
df["__y__"] = df[label_col].astype(str).map(label2id)

# Sanity-check: ensure no unmapped values
if df["__y__"].isna().any():
    bad = df.loc[df["__y__"].isna(), label_col].astype(str).value_counts().head(10)
    raise ValueError(f"Unmapped label values found. Examples:\n{bad}")

df["__y__"] = df["__y__"].astype(int)
print("Classes:", classes)


Classes: ['0', '1', '2', '3', '4', '5']


In [28]:
# Stratified split (Why: stable per-class distribution in train/val)
train_df, val_df = train_test_split(
    df[["__text__", "__y__"]],
    test_size=VAL_SIZE, random_state=SEED, stratify=df["__y__"]
)

In [29]:
print(f"Train={len(train_df)}, Val={len(val_df)}")

# ---------------------------
# Tokenizer
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

Train=507600, Val=56400


In [30]:
# def tokenize_batch(texts):
#     # Why padding in collator? → Faster, dynamic padding to the longest in the batch
#     return tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=False, return_tensors=None)
def tokenize_batch(texts):
    return tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=False, return_tensors=None)


train_enc = tokenize_batch(train_df["__text__"].tolist())
val_enc = tokenize_batch(val_df["__text__"].tolist())

In [31]:
# Convert to tensors for Trainer-compatible datasets
# class TextDS(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = {k: torch.tensor(v) for k, v in encodings.items()}
#         self.labels = torch.tensor(labels, dtype=torch.long)
#     def __len__(self): return len(self.labels)
#     def __getitem__(self, i):
#         item = {k: v[i] for k, v in self.encodings.items()}
#         item["labels"] = self.labels[i]
#         return item
class TextDS(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        # Keep raw python lists; do NOT convert whole arrays to tensors here
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        # Convert only the i-th sample to tensors; lengths can vary sample-to-sample
        item = {k: torch.tensor(v[i]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return item


train_ds = TextDS(train_enc, train_df["__y__"].values)
val_ds   = TextDS(val_enc,   val_df["__y__"].values)


In [32]:
# ---------------------------
# Class weights (Why: mitigate imbalance so rare classes matter)
# ---------------------------
counts = np.bincount(train_df["__y__"].values, minlength=num_labels).astype(np.float32)
weights = 1.0 / (counts + 1e-9)
weights = weights * (num_labels / weights.sum())  # normalize to mean≈1 for stable scaling
class_weights = torch.tensor(weights, dtype=torch.float)

In [40]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
)

# Custom Trainer to inject weighted loss
# class WeightedTrainer(Trainer):
#     def __init__(self, class_weights=None, **kwargs):
#         super().__init__(**kwargs)
#         self.class_weights = class_weights.to(self.args.device) if class_weights is not None else None
#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         logits = outputs.logits
#         loss_fct = nn.CrossEntropyLoss(weight=self.class_weights) if self.class_weights is not None else nn.CrossEntropyLoss()
#         loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
#         return (loss, outputs) if return_outputs else loss

# Custom Trainer to inject weighted loss (version-safe)
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights.to(self.args.device) if class_weights is not None else None

    # accept extra kwargs like num_items_in_batch for compatibility across transformers versions
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Pull labels but keep a copy of inputs for the forward pass
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights) if self.class_weights is not None else nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        if return_outputs:
            # Reattach labels so Trainer can use them downstream if needed
            outputs = (outputs, ) if not isinstance(outputs, tuple) else outputs
            return loss, outputs
        return loss



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
# Metrics (Why: macro-F1 is primary for imbalance; accuracy & micro-F1 for completeness)
# ---------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_micro": f1_score(labels, preds, average="micro"),
    }


In [42]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [43]:
import transformers
from transformers import TrainingArguments
from packaging import version

print("Transformers version:", transformers.__version__)

# Probe whether this install supports evaluation/save strategies
def _supports_strategies():
    try:
        # Will raise TypeError on legacy versions
        _ = TrainingArguments(output_dir="./_probe", evaluation_strategy="epoch", save_strategy="epoch")
        return True
    except TypeError:
        return False

IS_MODERN = _supports_strategies()

if IS_MODERN:
    # ✅ Modern transformers: full features enabled
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        logging_steps=50,
        fp16=USE_FP16,
        save_total_limit=2,
        report_to="none",

        # strategies + best-model-at-end
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        warmup_ratio=WARMUP_RATIO,
    )
    callbacks = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    print("Using MODERN training args (with eval/save strategies).")
else:
    # ⛳ Legacy transformers: KEEP IT SIMPLE
    # - No evaluation_strategy/save_strategy
    # - NO load_best_model_at_end
    # - NO early stopping callback
    print("Legacy transformers detected — using fallback training args (no eval/save strategies).")

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        logging_steps=50,
        fp16=USE_FP16,
        save_total_limit=2,

        # These exist in old versions; adjust if needed
        save_steps=500,          # checkpoint every N steps
        # eval_steps may or may not exist; we won't set it to avoid surprises
        # do NOT set report_to / warmup_ratio / load_best_model_at_end
    )
    callbacks = []  # no EarlyStoppingCallback on legacy


Transformers version: 4.57.0
Legacy transformers detected — using fallback training args (no eval/save strategies).


In [44]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=callbacks,
)


/tmp/ipython-input-1929889184.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


In [ ]:
# ---------------------------
# Train
# ---------------------------
train_result = trainer.train()
print("Best metrics so far:", trainer.state.best_metric)

Step,Training Loss
50,1.769700
100,1.468400


In [ ]:
eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)

pred = trainer.predict(val_ds)
y_true = val_df["__y__"].values
y_pred = np.argmax(pred.predictions, axis=1)

print("\nClassification report (per class):")
print(classification_report(y_true, y_pred, target_names=[id2label[i] for i in range(num_labels)], digits=4))

print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred))
